In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 2A: terminal-latent covariance six-arm interface

This entry reuses the cell order of the completed public-statistic run08 notebook: Drive mount, GitHub fetch, dependency setup, fixed input/output, then a cancellable subprocess. It replaces only the old OFF1/OFF2/Y_MINUS feedback experiment with one ordinary Wan terminal generation followed by C2A OFF/ZERO/+rho e1/-rho e1/+rho e2/-rho e2.

In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_2a_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
# Keep Colab CUDA PyTorch; actual versions are recorded by the C2A result.


## Fixed first interface configuration

The C2A configuration carries forward the prior actual Wan entry: Wan2.1 T2V 1.3B, cube prompt, seed 1275, 320x512, 49 frames, 50 steps and CFG 5. It uses one terminal latent, not a feedback prefix or a second generated control.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_minimal_run.json'
RUN_ID = 'c2a_2a_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_2A') / RUN_ID
RUN_2A = False
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
if RUN_2A:
    import os, signal
    command = [sys.executable, '-m', 'runtime.c2a.run_2a', '--config', str(CONFIG), '--output', str(OUTPUT), '--execute']
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True)
    try:
        returncode = process.wait()
    except BaseException:
        try: process.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        raise
    print('launcher exit', returncode)
    print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
    if returncode: raise subprocess.CalledProcessError(returncode, command)
else:
    print('Prepared only: RUN_2A is False.')


## Persisted result

An enabled run writes directly to `MyDrive/Video-WM/C2A_2A/<UTC-run-id>/`: `config.json`, `result.json`, all completed arm MP4 files and every retained arm/setup failure. It performs no automatic retry. The first interface run measures quality and signed two-axis response; it does not claim a threshold PASS, fourth-public-point calibration, a full packet, or a completed blind detector.